# Feature Engineering

Create advanced features for Marketing Mix Modeling.

In [4]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

input_path = Path('../data/processed/cleaned_data.csv')
output_path = Path('../data/processed/feature_engineered_data.csv')

print(f'Loading cleaned data from: {input_path}')
df = pd.read_csv(input_path)

# Helper functions

def _sort_for_time_series(df, time_col='Week', group_cols=None):
    df2 = df.copy()
    if time_col in df2.columns:
        df2[time_col] = pd.to_datetime(df2[time_col], errors='coerce')
    sort_cols = []
    if group_cols:
        sort_cols.extend([c for c in group_cols if c in df2.columns])
    if time_col in df2.columns:
        sort_cols.append(time_col)
    if sort_cols:
        df2 = df2.sort_values(sort_cols)
    return df2


def _to_numeric_flag(series):
    if series.dtype == bool:
        return series.astype(int)
    s = series.astype(str).str.lower().str.strip()
    return s.map({'true':1,'false':0,'1':1,'0':0}).fillna(0).astype(int)


def create_total_media_exposure(df, media_columns=None, output_col='Total_Media_Exposure'):
    media_columns = media_columns or [
        'TV_Impressions','YouTube_Impressions','Facebook_Impressions',
        'Instagram_Impressions','Print_Readership','Radio_Listenership'
    ]
    cols = [c for c in media_columns if c in df.columns]
    out = df.copy()
    if not cols:
        out[output_col] = 0.0
        return out
    out[output_col] = out[cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    return out


def create_promotion_score(df, promotion_columns=None, spend_column='Trade_Spend', output_col='Promotion_Score'):
    promotion_columns = promotion_columns or ['Feature_Flag','Display_Flag','TPR_Flag']
    out = df.copy()
    flags = [c for c in promotion_columns if c in out.columns]
    flag_score = pd.Series(0.0, index=out.index)
    if flags:
        flag_score = out[flags].apply(_to_numeric_flag).mean(axis=1)
    if spend_column in out.columns:
        spend = pd.to_numeric(out[spend_column], errors='coerce').fillna(0)
        s = np.log1p(spend)
        smin, smax = s.min(), s.max()
        if smax > smin:
            s = (s - smin) / (smax - smin)
        else:
            s = pd.Series(0.0, index=out.index)
    else:
        s = pd.Series(0.0, index=out.index)
    out[output_col] = 100.0 * (0.5 * flag_score + 0.5 * s)
    return out


def create_lagged_features(df, columns, lags=[1,2,4], group_cols=None, time_col='Week'):
    df2 = _sort_for_time_series(df, time_col=time_col, group_cols=group_cols)
    avail = [c for c in columns if c in df2.columns]
    if group_cols:
        keys = [c for c in group_cols if c in df2.columns]
        grouped = df2.groupby(keys, sort=False)
    else:
        grouped = None
    for col in avail:
        for lag in lags:
            name = f'{col}_lag_{lag}'
            if grouped is not None:
                df2[name] = grouped[col].shift(lag)
            else:
                df2[name] = df2[col].shift(lag)
    return df2


def create_rolling_features(df, columns, window_sizes=[4,12], group_cols=None, time_col='Week'):
    df2 = _sort_for_time_series(df, time_col=time_col, group_cols=group_cols)
    avail = [c for c in columns if c in df2.columns]
    if group_cols:
        keys = [c for c in group_cols if c in df2.columns]
        grouped = df2.groupby(keys, sort=False)
    else:
        grouped = None
    for col in avail:
        for w in window_sizes:
            mean_col = f'{col}_rolling_mean_{w}'
            std_col = f'{col}_rolling_std_{w}'
            if grouped is not None:
                df2[mean_col] = grouped[col].transform(lambda s: s.rolling(window=w, min_periods=1).mean())
                df2[std_col] = grouped[col].transform(lambda s: s.rolling(window=w, min_periods=1).std())
            else:
                df2[mean_col] = df2[col].rolling(window=w, min_periods=1).mean()
                df2[std_col] = df2[col].rolling(window=w, min_periods=1).std()
    return df2


def create_interaction_features(df, pairs=None):
    df2 = df.copy()
    pairs = pairs or [
        ('Total_Media_Exposure','Promotion_Score'),
        ('Trade_Spend','Total_Media_Exposure'),
        ('Trade_Spend','Promotion_Score'),
        ('TV_Impressions','YouTube_Impressions')
    ]
    for a,b in pairs:
        if a in df2.columns and b in df2.columns:
            left = pd.to_numeric(df2[a], errors='coerce').fillna(0)
            right = pd.to_numeric(df2[b], errors='coerce').fillna(0)
            df2[f'{a}_x_{b}'] = left * right
    return df2


def prepare_feature_importance_data(df, target_col='Sales_Value', id_columns=None):
    id_columns = id_columns or ['Week','Geo','Brand','SKU']
    out = df.copy()
    for c in out.select_dtypes(include=['bool']).columns:
        out[c] = out[c].astype(int)
    numeric = out.select_dtypes(include=[np.number]).copy()
    if target_col not in numeric.columns and target_col in out.columns:
        numeric[target_col] = pd.to_numeric(out[target_col], errors='coerce')
    if target_col not in numeric.columns:
        raise ValueError(f"Target column '{target_col}' is required")
    feature_columns = [c for c in numeric.columns if c != target_col and c not in id_columns]
    return numeric[feature_columns + [target_col]].copy(), feature_columns

# Build pipeline inline
print('Creating Total Media Exposure...')
df = create_total_media_exposure(df)
print('Creating Promotion Score...')
df = create_promotion_score(df)

lag_cols = ['Sales_Value','Total_Media_Exposure','Promotion_Score','Trade_Spend']
print('Creating lagged features...')
df = create_lagged_features(df, lag_cols, lags=[1,2,4], group_cols=['Geo','Brand','SKU'])

roll_cols = ['Sales_Value','Total_Media_Exposure','Promotion_Score','Trade_Spend']
print('Creating rolling features...')
df = create_rolling_features(df, roll_cols, window_sizes=[4,12], group_cols=['Geo','Brand','SKU'])

print('Creating interaction features...')
df = create_interaction_features(df)

print('Preparing feature-importance dataset...')
fi_df, fi_cols = prepare_feature_importance_data(df, target_col='Sales_Value', id_columns=['Week','Geo','Brand','SKU'])

# Save final dataset
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print(f'Feature-engineered dataset saved to: {output_path} (shape: {df.shape})')

Loading cleaned data from: ..\data\processed\cleaned_data.csv
Creating Total Media Exposure...
Creating Promotion Score...
Creating lagged features...
Creating rolling features...
Creating interaction features...
Preparing feature-importance dataset...
Feature-engineered dataset saved to: ..\data\processed\feature_engineered_data.csv (shape: (11232, 62))
